# Indice denso (bge-m3 + FAISS) en Colab y en local

Flujo: **A (local)** empaqueta -> **B (Colab, GPU)** codifica -> **C (local)** integra el resultado.
Las celdas de cada parte dicen donde correrlas. Una sola configuracion congelada: el `cache_emb` final
debe venir **entero de un solo dispositivo** (no mezclar shards de MPS y CUDA: difieren ~1e-6 y pueden cambiar empates).

En Colab: Entorno de ejecucion -> Cambiar tipo -> **GPU T4**.

## A. LOCAL: crear `colab_paquete.zip` (correr desde la raiz del repo)

In [ ]:
# LOCAL. Empaqueta solo lo necesario para construir_indice.py.
import zipfile, hashlib
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent  # si el notebook corre desde notebooks/
assert (ROOT / "src" / "config.py").exists(), "correr desde la raiz del repo"
ESPERADO = "d133b9906b0ea50e2bdaee938914fc60bb24322022aa6fb67961bd74bb2d27ab"
chunks = ROOT / "data_corpus" / "indice" / "chunks.jsonl"
sha = hashlib.sha256(chunks.read_bytes()).hexdigest()
assert sha == ESPERADO, f"chunks.jsonl distinto del congelado: {sha}"
archivos = ["src/config.py", "src/indexacion/construir_indice.py", "src/recuperacion/lexico.py",
            "scripts/citations.py", "requirements.txt", "data_corpus/indice/chunks.jsonl"]
salida = ROOT / "data_corpus" / "colab_paquete.zip"
with zipfile.ZipFile(salida, "w", zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(ROOT / a, a)
print(salida, round(salida.stat().st_size / 1e6, 1), "MB")

## B. COLAB: construir el indice (GPU)

In [ ]:
# COLAB. Verifica GPU y sube colab_paquete.zip (data_corpus/colab_paquete.zip en tu maquina).
import torch
assert torch.cuda.is_available(), "Activar GPU: Entorno de ejecucion -> Cambiar tipo -> T4"
print(torch.cuda.get_device_name(0))
from google.colab import files
subido = files.upload()          # elegir colab_paquete.zip

In [ ]:
# COLAB. Descomprimir e instalar (torch ya viene en Colab; no se reinstala).
!unzip -oq colab_paquete.zip -d /content/syntax
!pip -q install sentence-transformers==6.1.0 bm25s==0.3.11 faiss-cpu==1.15.1 numpy

In [ ]:
# COLAB. Comprobar que el corpus es el congelado antes de codificar.
import hashlib
sha = hashlib.sha256(open("/content/syntax/data_corpus/indice/chunks.jsonl", "rb").read()).hexdigest()
assert sha == "d133b9906b0ea50e2bdaee938914fc60bb24322022aa6fb67961bd74bb2d27ab", sha
print("chunks.jsonl ok")

In [ ]:
# COLAB. Codificar (~35.105 fragmentos). Cada 512 se guarda un shard; si se corta, volver a correr esta celda.
# Antes: !nvidia-smi debe mostrar la GPU casi libre (si no, reiniciar la sesion). Si hay OOM, bajar --batch a 8.
%cd /content/syntax
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True SYNTAX_DEVICE=cuda PYTHONUNBUFFERED=1 python src/indexacion/construir_indice.py --batch 16 --no-manifest

In [ ]:
# COLAB. Empaquetar y descargar el resultado.
!cd /content/syntax/data_corpus && zip -qr /content/resultado_indice.zip cache_emb indice/index.faiss indice/indice_info.json
from google.colab import files
files.download("/content/resultado_indice.zip")

## C. LOCAL: integrar el resultado de Colab

1. Si el proceso local de MPS sigue corriendo, detenerlo **solo si vas a usar Colab** (no mezclar).
2. Mover `resultado_indice.zip` a `data_corpus/`.
3. Correr la celda siguiente: reemplaza `cache_emb` por el de Colab y reconstruye desde cache (segundos, sin encoder).

In [ ]:
# LOCAL. Reemplaza cache_emb local por el de Colab y verifica.
import shutil, zipfile, subprocess, sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src" / "config.py").exists() else Path.cwd().parent
DC = ROOT / "data_corpus"
zip_res = DC / "resultado_indice.zip"
assert zip_res.exists(), f"falta {zip_res}"
if (DC / "cache_emb").exists():
    shutil.move(DC / "cache_emb", DC / "cache_emb_mps_parcial")   # respaldo; borrar luego
zipfile.ZipFile(zip_res).extractall(DC)
env = {**os.environ, "SYNTAX_DEVICE": "cpu", "PYTHONUNBUFFERED": "1"}
# Todo esta en cache: no codifica nada. Sin --no-manifest actualiza n_fragmentos/sha256 del manifest.
subprocess.run([sys.executable, str(ROOT / "src/indexacion/construir_indice.py")], env=env, check=True, cwd=ROOT)

## D. LOCAL alternativo (sin Colab): construir con MPS/CUDA/CPU

```bash
SYNTAX_DEVICE=mps python src/indexacion/construir_indice.py --batch 16     # Mac, ~4 frag/s
SYNTAX_DEVICE=cuda python src/indexacion/construir_indice.py --batch 16    # GPU del campus
```
La caché permite retomar tras un corte. Luego medir:
`python src/evaluacion/retrieval_eval.py --modo bm25 denso hibrido --experimento e01_bge_m3`

## E. Trabajo en paralelo (varias personas)

Cada persona codifica un trozo distinto con `--particion I/N` (los trozos son disjuntos y no dependen de la cache local).
**Todos deben usar el mismo tipo de dispositivo (CUDA, p. ej. un Colab cada uno)**: mezclar MPS y CUDA mete diferencias ~1e-6 en el indice.

| Persona | Comando de codificacion (celda E1) |
|---|---|
| 1 | `--particion 1/3` |
| 2 | `--particion 2/3` |
| 3 | `--particion 3/3` |

Pasos: (1) cada uno hace A y B hasta instalar y verificar el sha256; (2) en vez de la celda de codificacion, corre **E1** con su trozo;
(3) **E2** descarga `trozo_I_de_N.zip`; (4) una persona junta los tres zips en `data_corpus/` y corre **E3**.

In [ ]:
# COLAB (E1). Cambiar I y N. Solo escribe shards en cache_emb; no construye index.faiss.
I, N = 1, 3
%cd /content/syntax
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True SYNTAX_DEVICE=cuda PYTHONUNBUFFERED=1 python src/indexacion/construir_indice.py --particion {I}/{N} --batch 16

In [ ]:
# COLAB (E2). Empaquetar y descargar los shards de este trozo.
!cd /content/syntax/data_corpus && zip -qr /content/trozo_{I}_de_{N}.zip cache_emb
from google.colab import files
files.download(f"/content/trozo_{I}_de_{N}.zip")

In [ ]:
# LOCAL (E3). Junta los trozo_*_de_*.zip de data_corpus/ y construye el indice desde cache (sin codificar).
import shutil, zipfile, subprocess, sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src" / "config.py").exists() else Path.cwd().parent
DC = ROOT / "data_corpus"
zips = sorted(DC.glob("trozo_*_de_*.zip"))
assert zips, f"no hay trozo_*_de_*.zip en {DC}"
if (DC / "cache_emb").exists():
    shutil.move(DC / "cache_emb", DC / "cache_emb_mps_parcial")   # no mezclar dispositivos
for z in zips:
    zipfile.ZipFile(z).extractall(DC)   # los shards de cada trozo se suman en cache_emb/ (nombres unicos)
print(len(zips), "trozos integrados")
env = {**os.environ, "SYNTAX_DEVICE": "cpu", "PYTHONUNBUFFERED": "1"}
# Guarda: si falta algun trozo abortar, en vez de codificar el resto en CPU durante horas.
sys.path.insert(0, str(ROOT / "src"))
from indexacion import construir_indice as ci
cache = ci.cargar_cache()
faltan = sum(1 for c in ci.cargar_chunks() if ci.sha256_bytes(c["retrieval_text"].encode("utf-8")) not in cache)
assert faltan == 0, f"faltan {faltan} vectores: falta algun trozo"
subprocess.run([sys.executable, str(ROOT / "src/indexacion/construir_indice.py")], env=env, check=True, cwd=ROOT)